<a href="https://colab.research.google.com/github/lbrugola/dmeyf2026/blob/main/494_TareaHogar_04_Etapa_HT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea para el Hogar 04

##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion sw Hiperparámetros lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.


Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 29 04:35:10 AM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1940002,103.7,3543515,189.3,3543515,189.3
Vcells,16453645,125.6,111549456,851.1,102906040,785.2


### 2.2.2 Carga de Librerias

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [ ]:
PARAM <- list()
PARAM$experimento <- 5940

PARAM$semilla_primigenia <- 247067

In [ ]:
PARAM$kaggle$competencia <- "utn-2026-inicial"
PARAM$kaggle$cortes <- seq(9000, 12000, by= 500)

In [ ]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5

In [ ]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 1.0, # scale_pos_weight > 0.0

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 100,
  learning_rate= 0.10,  # >=0
  feature_fraction= 1.0, # 0 < ff <= 1.0
  num_leaves= 32, # integer >= 2
  min_data_in_leaf= 20 # integer >= 0
)


### 2.2.4  Preprocesamiento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [ ]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [ ]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

head(dataset_train)

numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,⋯,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo,clase_ternaria,clase01,azar,training
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<chr>,<int>,<dbl>,<int>
11673459,202107,1,0,0,56,138,2686.85,29431.10,688.94,⋯,0.00,4178,15732.34,1,0,1137.81,CONTINUA,0,0.5405725,0
11673463,202107,1,0,0,49,106,501.84,11441.64,221.67,⋯,NA,NA,NA,NA,NA,NA,CONTINUA,0,0.2063838,1
11674064,202107,1,0,0,60,59,-2941.46,-8218.09,2929.35,⋯,0.00,1754,11140.52,3,0,21524.55,CONTINUA,0,0.5947538,0
11674166,202107,1,0,0,46,279,17318.57,159583.04,731.86,⋯,16.33,2244,1241.17,4,0,1313.76,CONTINUA,0,0.7235399,0
11674316,202107,1,0,0,47,198,2551.54,30095.13,2088.76,⋯,0.00,6023,0.00,0,0,1313.76,CONTINUA,0,0.0398285,1
11674337,202107,1,0,0,69,264,4117.61,62102.58,4934.07,⋯,0.00,3302,49080.20,21,0,5372.34,CONTINUA,0,0.8164479,0


In [ ]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 83235

[1] 154

2.2.5 Configuracion del Grid Search

In [ ]:
Estimar_AUC_lightgbm <- function(x) {

  # Registramos el tiempo inicial
  t0 <- Sys.time()

  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # Entrenamos con Cross-Validation
  modelocv <- lgb.cv(
    data = dtrain,
    nfold = PARAM$hyperparametertuning$xval_folds,
    stratified = TRUE,
    param = param_completo,
    eval_train_metric = TRUE,
    early_stopping_rounds = 50
  )

  # Registramos el tiempo final y calculamos la duración en segundos
  t1 <- Sys.time()
  tiempo_segundos <- as.numeric(difftime(t1, t0, units = "secs"))

  # extraemos best iter y AUC para train y validation set, luego calculamos el gap
  best_iter <- modelocv$best_iter
  auc_val <- modelocv$record_evals$valid$auc$eval[[best_iter]]
  auc_train <- modelocv$record_evals$train$auc$eval[[best_iter]]
  gap_overfitting <- auc_train - auc_val

  message(
    format(Sys.time(), "%a %b %d %X %Y "),
    toString(x),
    " | Best Iter: ", best_iter,
    " | AUC Train: ", round(auc_train, 4),
    " | AUC Val: ", round(auc_val, 4),
    " | Overfitting Gap: ", round(gap_overfitting, 4),
    " | Tiempo: ", round(tiempo_segundos, 1), " seg"
  )

  rm(modelocv)
  gc(full = TRUE, verbose = FALSE)

  return(list(
    AUC = auc_val,
    best_iter = best_iter,
    AUC_train = auc_train,
    gap = gap_overfitting,
    tiempo_seg = round(tiempo_segundos, 2)
  ))
}

In [ ]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
# Definición del archivo de checkpoint dentro de la carpeta del experimento
archivo_checkpoint <- "tb_grid_search_checkpoint.txt"

# Generamos el producto cartesiano de hiperparámetros a probar
# Experimento 1
tb_nueva <- CJ(
  num_iterations   = c(50, 100, 500,1000),
  learning_rate    = c(0.001, 0.01, 0.1),
  num_leaves       = c(10, 50, 100, 500),
  feature_fraction = c(0.4,0.5,0.8),
  min_data_in_leaf = c(100,500,1000,5000),
  lambda_l1 = c(0,1),
  lambda_l2 = c(0)
)

# Experimento 2
# tb_nueva <- CJ(
#   num_iterations   = c(100,300,500),
#   learning_rate    = c(0.01, 0.05, 0.1),
#   num_leaves       = c(50, 100, 500),
#   min_data_in_leaf = c(20,500,1000),
#   min_gain_to_split = c(0, 0.01, 0.1),
#   bagging_fraction = c(0.7, 0.9),
#   bagging_freq     = c(1),
#   feature_fraction = c(0.8),
#   lambda_l1        = c(0, 1)
# )

# Control de reanudación (Checkpoint)
if (file.exists(archivo_checkpoint)) {
  message("--> Cargando checkpoint previo...")
  tb_procesada <- fread(archivo_checkpoint)
  cols_hiperpar <- colnames(tb_nueva)
  tb_pendiente <- tb_nueva[!tb_procesada, on = cols_hiperpar]
  message("Pendientes por procesar: ", nrow(tb_pendiente))
} else {
  message("--> Iniciando Grid Search desde cero...")
  tb_procesada <- data.table()
  tb_pendiente <- copy(tb_nueva)
}

# Bucle de procesamiento de combinaciones pendientes
if (nrow(tb_pendiente) > 0) {

  for (i in 1:nrow(tb_pendiente)) {

    params_i <- as.list(tb_pendiente[i, ])
    res <- Estimar_AUC_lightgbm(params_i)

    # Ensamblamos la fila completa agregando 'best_score'
    fila_resultado <- cbind(
      tb_pendiente[i, ],
      data.table(
        AUC_val    = res$AUC_val,     # Métricas en Validación / Test
        AUC_train  = res$AUC_train,   # Métricas en Entrenamiento
        best_score = res$best_score,  # Métricas del best_score nativo
        best_iter  = res$best_iter,
        gap        = res$gap,
        tiempo_seg = res$tiempo_seg
      )
    )

    # Guardamos en disco
    fwrite(
      fila_resultado,
      file = archivo_checkpoint,
      sep = "\t",
      append = TRUE
    )
  }
}

# Consolidación de resultados
tb_grid_completa <- fread(archivo_checkpoint)
setorder(tb_grid_completa, -AUC_val) # Ordenamos por el mejor AUC de Test

# Imprimimos la tabla ordenada por rendimiento en Test
print(tb_grid_completa)

In [ ]:
fwrite( tb_grid_completa,
  file= "tb_grid_serach_01.txt",
  sep="\t",
  append= TRUE
)

In [ ]:
write_yaml( PARAM, file="PARAM.yml")